In [15]:
import os
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
import joblib

In [ ]:
# Load and Prepare WESAD
def load_wesad_subject(path):
    with open(path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    
    # Wrist signals (from Empatica E4)
    wrist = data['signal']['wrist']
    bvp   = wrist['BVP'].flatten()   # 64 Hz → derive HR
    eda   = wrist['EDA'].flatten()   # 4 Hz
    temp  = wrist['TEMP'].flatten()  # 4 Hz
    acc   = wrist['ACC']             # 32 Hz, 3-axis

    # Labels: 0=not defined, 1=baseline, 2=stress, 3=amusement
    labels = data['label'].flatten()

    return bvp, eda, temp, acc, labels

# Keep only baseline (1) and stress (2) for binary classification
def filter_labels(labels):
    mask = np.isin(labels, [1, 2])
    binary = np.where(labels == 2, 1, 0)  # stress=1, baseline=0
    return mask, binary


In [10]:
# Feature Extraction
# Use a sliding window (common: 60s window, 0.25s step) and extract features per window.
#All signals should be downsampled or upsampled to a common rate first.

def extract_features(eda_window, hr_window, temp_window):
    feats = {}

    # --- EDA features ---
    feats['eda_mean']     = np.mean(eda_window)
    feats['eda_std']      = np.std(eda_window)
    feats['eda_min']      = np.min(eda_window)
    feats['eda_max']      = np.max(eda_window)
    feats['eda_range']    = np.ptp(eda_window)
    feats['eda_skew']     = skew(eda_window)
    feats['eda_kurt']     = kurtosis(eda_window)

    # EDA frequency domain (SCR component lives in 0.05–0.25 Hz)
    freqs, psd = welch(eda_window, fs=4)
    scr_band   = psd[(freqs >= 0.05) & (freqs <= 0.25)]
    feats['eda_scr_power'] = np.sum(scr_band)

    # --- HR features ---
    feats['hr_mean']  = np.mean(hr_window)
    feats['hr_std']   = np.std(hr_window)
    feats['hr_min']   = np.min(hr_window)
    feats['hr_max']   = np.max(hr_window)
    feats['hr_range'] = np.ptp(hr_window)

    # HRV proxy: RMSSD from successive differences
    diffs = np.diff(hr_window)
    feats['hr_rmssd'] = np.sqrt(np.mean(diffs**2))

    # --- Temperature features ---
    feats['temp_mean']  = np.mean(temp_window)
    feats['temp_slope'] = np.polyfit(range(len(temp_window)), temp_window, 1)[0]

    return feats


def build_feature_matrix(bvp, eda, temp, labels, fs_eda=4, fs_temp=4,
                           window_s=60, step_s=0.25):
    win_eda  = int(window_s * fs_eda)
    step_eda = max(1, int(step_s * fs_eda))

    rows, y = [], []
    for start in range(0, len(eda) - win_eda, step_eda):
        end = start + win_eda

        # Map eda index → label index (labels are at 700 Hz in WESAD)
        label_idx = int(start / fs_eda * 700)
        label     = labels[label_idx] if label_idx < len(labels) else -1
        if label not in [0, 1]:
            continue

        # For HR: derive from BVP (simple: use BVP mean as HR proxy,
        # or run a proper peak-detection pipeline)
        bvp_start = int(start / fs_eda * 64)
        bvp_end   = int(end   / fs_eda * 64)
        hr_window = bvp[bvp_start:bvp_end]

        temp_window = temp[start:end]
        eda_window  = eda[start:end]

        feats = extract_features(eda_window, hr_window, temp_window)
        rows.append(feats)
        y.append(label)

    return pd.DataFrame(rows), np.array(y)


In [19]:
# ================= MAIN EXECUTION =================

wesad_dir = "WESAD/"
wesad_subject_paths = {}

# 1. FIX: Dynamically find .pkl files inside each S* subject folder
for subject_folder in Path(wesad_dir).iterdir():
    if subject_folder.is_dir() and subject_folder.name.startswith('S'):
        # The .pkl file is named exactly like the folder (e.g., S2/S2.pkl)
        pkl_path = subject_folder / f"{subject_folder.name}.pkl"
        if pkl_path.exists():
            wesad_subject_paths[subject_folder.name] = str(pkl_path)
        else:
            print(f"Warning: {pkl_path} not found.")

if not wesad_subject_paths:
    raise FileNotFoundError(f"No subject folders with .pkl files found in '{wesad_dir}'. Please check your dataset path.")

print(f"Found {len(wesad_subject_paths)} subjects: {list(wesad_subject_paths.keys())}")

# Build feature matrix across all subjects
all_X, all_y, groups = [], [], []

for subj_id, path in wesad_subject_paths.items():
    print(f"Processing {subj_id}...")
    bvp, eda, temp, acc, labels = load_wesad_subject(path)
    
    # 2. FIX: Pass the FULL arrays. Filtering is handled safely 
    # inside build_feature_matrix to prevent index misalignment.
    X, y = build_feature_matrix(bvp, eda, temp, labels)
    
    if len(X) > 0:
        all_X.append(X)
        all_y.append(y)
        groups.extend([subj_id] * len(y))
    else:
        print(f"Warning: No valid windows extracted for {subj_id}.")

X_all = pd.concat(all_X).reset_index(drop=True)
y_all = np.concatenate(all_y)
groups = np.array(groups)

# Leave-One-Subject-Out CV — critical for generalization
logo = LeaveOneGroupOut()

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(n_estimators=200, random_state=42))
])

scores = cross_val_score(pipe, X_all, y_all, cv=logo, groups=groups,
                          scoring='f1_weighted')
print(f"\nLOSO F1: {scores.mean():.3f} ± {scores.std():.3f}")

# Train final model on all WESAD subjects
pipe.fit(X_all, y_all)
joblib.dump(pipe, 'wesad_stress_classifier.pkl')
print("Model successfully saved to 'wesad_stress_classifier.pkl'")

Found 15 subjects: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Processing S10...


/home/jovyan/.local/lib/python3.10/site-packages/scipy/signal/_spectral_py.py:790: UserWarning: nperseg = 256 is greater than input length  = 240, using nperseg = 240
  freqs, _, Pxy = _spectral_helper(x, y, fs, window, nperseg, noverlap,


Processing S11...
Processing S13...
Processing S14...
Processing S15...
Processing S16...
Processing S17...
Processing S2...
Processing S3...
Processing S4...
Processing S5...
Processing S6...
Processing S7...
Processing S8...
Processing S9...

LOSO F1: 0.724 ± 0.112
Model successfully saved to 'wesad_stress_classifier.pkl'


In [ ]:
# Sync EmbracePlus Data with Stroop Trial Timestamps
# Stroop script already saves trial_timestamp per trial. 
# Match this to EmbracePlus export.


def load_and_sync(fitbit_hr_path, fitbit_eda_path, stroop_csv_path):
    # EmbracePlus exports typically give HR as 1-min averages or 5s intervals
    hr_df  = pd.read_csv(fitbit_hr_path, parse_dates=['Time'])
    eda_df = pd.read_csv(fitbit_eda_path, parse_dates=['timestamp'])

    stroop = pd.read_csv(stroop_csv_path, parse_dates=['trail_timestamp'])

    # Label each row with block condition
    stroop['stress_label'] = stroop['block'].map(
        {'baseline': 0, 'stress': 1, 'practice': np.nan}
    )

    # For each trial, get the nearest HR/EDA reading
    def nearest_signal(ts, signal_df, time_col, val_col, tolerance_s=30):
        delta = (signal_df[time_col] - ts).abs()
        idx   = delta.idxmin()
        if delta[idx].total_seconds() <= tolerance_s:
            return signal_df.loc[idx, val_col]
        return np.nan

    stroop['hr']  = stroop['trail_timestamp'].apply(
        lambda t: nearest_signal(t, hr_df,  'Time',      'Value'))
    stroop['eda'] = stroop['trail_timestamp'].apply(
        lambda t: nearest_signal(t, eda_df, 'timestamp', 'eda_value'))

    return stroop


In [ ]:
# Apply / Fine-tune on Your Data
model = joblib.load('wesad_stress_classifier.pkl')

# Option A: Direct inference (no labels from your experiment needed)
# Extract features from your Fitbit data in same window format → predict
your_X = extract_your_fitbit_features(synced_df)  # same feature names as WESAD
your_preds = model.predict(your_X)

# Option B: Fine-tune with Stroop block as pseudo-ground-truth
# baseline block → label 0, stress block → label 1
from sklearn.base import clone

fine_tuned = clone(model)
fine_tuned.fit(your_X_labeled, your_y_labeled)

# Option C: Compare predictions to block labels for validation
from sklearn.metrics import classification_report
print(classification_report(your_y_labeled, your_preds))
